# Session 14 lab: the machine learning workflow

You will prepare the Nile Microfinance loans (fictional data), split them into a training set and a test set, fit a baseline model and a first model, and read what accuracy hides.

In every step: **PREDICT**, run, **READ**, **CHANGE** one thing. Cells with `# your code here` hold a placeholder: replace that line with your answer. Run the cells in order. Before you answer, a `check(...)` line prints NOT YET. It prints OK when your number matches. Run the last cell to count the checks.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

# Run this cell once. It lets a check say NOT YET instead of stopping the notebook when your answer is still empty.
def safe(f):
    try:
        return f()
    except Exception:
        return np.nan

---
## Step 1: Look at the loans

Each record of loans.csv is one loan. The target is default: 1 if the borrower stopped repaying (the positive class), 0 if the loan was repaid.

**PREDICT first.** What share of the 8,000 loans do you think defaulted: about 1 percent, 12 percent or 50 percent?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
loans = pd.read_csv("loans.csv")
print(loans.shape)
default_rate = np.nan   # your code here: the mean of the default column is the share of 1s
print(default_rate)
print(loans.default.value_counts())

check("8,000 loans", len(loans), 8000, 0)
check("default rate", safe(lambda: default_rate), 0.1255, 0.001)

**READ the output.** Which class is the positive class, and is the data balanced?

> My answer: 

---
## Step 2: Prepare: impossible values

A borrower cannot be 0 or 199 years old. Mark ages under 18 or over 100, and replace them with the median of the valid ages.

**PREDICT first.** Print the youngest and oldest ages with loans.age.sort_values(). How many impossible ages do you expect?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
print(loans.age.sort_values().head(3).tolist(), loans.age.sort_values().tail(3).tolist())
bad_age = pd.Series(False, index=loans.index)   # your code here: (loans.age < 18) | (loans.age > 100), with brackets round each test
age_median = np.nan   # your code here: the median age of the rows where bad_age is False
loans.loc[bad_age, "age"] = age_median
print(bad_age.sum(), "ages replaced by", age_median)

check("impossible ages", safe(lambda: int(bad_age.sum())), 2, 0)
check("median of valid ages", safe(lambda: age_median), 36, 0)

**READ the output.** Why use the median of the valid ages? What would the mean of all ages, including 199, do?

> My answer: 

---
## Step 3: Prepare: missing values with a missing-indicator

Make the flag income_missing first. Then fill the blank incomes with the median income. The order matters.

**PREDICT first.** About how many incomes are blank: 40, 400 or 4,000?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
loans["income_missing"] = 0   # your code here: 1 where the income is blank, 0 elsewhere (.isna().astype(int))
median_income = np.nan   # your code here: the median of monthly_income_ugx (blanks are skipped)
loans["monthly_income_ugx"] = loans.monthly_income_ugx.fillna(median_income)
print(loans.income_missing.sum(), "flagged;", loans.monthly_income_ugx.isna().sum(), "blanks left; median", median_income)

check("blank incomes flagged", safe(lambda: int(loans.income_missing.sum())), 379, 0)
check("no blanks left", safe(lambda: int(loans.monthly_income_ugx.isna().sum())), 0, 0)
check("median income", safe(lambda: median_income), 642600.0, 1)

**CHANGE one thing.** Reload the file in a new cell and fill the blanks BEFORE making the flag. What does income_missing.sum() give now?

> What I saw: 

---
## Step 4: Prepare: feature engineering

Build the instalment-to-income ratio (also called the payment-to-income ratio): the monthly instalment as a share of monthly income. The instalment is the loan amount times 1.15 divided by the term in months. The 1.15 is an illustrative assumption for interest and fees. It is not in the data.

**PREDICT first.** Will the default rate be higher in the group of loans with the highest ratio? By how much?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
loans["instalment_to_income"] = np.nan   # your code here: loan_amount_ugx * 1.15 / term_months / monthly_income_ugx
top = loans.instalment_to_income > loans.instalment_to_income.quantile(0.75)
top_rate = np.nan   # your code here: the default rate among the loans where top is True
print("median ratio", round(loans.instalment_to_income.median(), 3), " default rate in the highest quarter", round(top_rate, 3))

check("median ratio", safe(lambda: loans.instalment_to_income.median()), 0.2916, 0.005)
check("default rate in the highest quarter of the ratio", safe(lambda: top_rate), 0.2775, 0.01)

**READ the output.** Compare top_rate with the overall default rate. Would this ratio help a model?

> My answer: 

---
## Step 5: Choose the features and the target

Twelve features known at approval. collection_calls is left out on purpose: it is recorded after a loan goes bad, so it leaks the answer.

In [ ]:
features = ["age", "monthly_income_ugx", "income_missing", "loan_amount_ugx", "term_months", "previous_loans",
            "previous_late_payments", "group_loan", "savings_balance_ugx", "has_mobile_money", "distance_to_branch_km", "instalment_to_income"]
X = loans[["age"]]   # your code here: the feature columns of loans (the list features)
y = pd.Series(0, index=loans.index)   # your code here: the target column
print(X.shape, y.shape)

check("12 features", X.shape[1], 12, 0)
check("no blanks in X", safe(lambda: int(X.isna().sum().sum())), 0, 0)

---
## Step 6: Split into a training set and a test set

Hold back 25 percent of the loans. stratify=y keeps the default rate the same in both parts, and random_state=42 makes the split repeatable. Note: the medians above were computed on all 8,000 loans before the split. That is a small leakage risk that we accept on purpose for teaching.

**PREDICT first.** How many defaults will the test set hold if it keeps the same rate as the whole file?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
X_train, X_test, y_train, y_test = X.iloc[:5], X.iloc[5:10], y.iloc[:5], y.iloc[5:10]   # your code here: train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(len(X_train), len(X_test), "loans;", y_train.sum(), "and", y_test.sum(), "defaults")
print("first test rows:", X_test.index[:10].tolist())   # the same rows as the course notes when random_state=42 and stratify=y

check("test loans", len(X_test), 2000, 0)
check("defaults in the test set", int(y_test.sum()), 251, 0)
check("defaults in the training set", int(y_train.sum()), 753, 0)

**CHANGE one thing.** The prepared file filled the blank incomes with the median of all 8,000 loans. Compute the median income of the training loans only (use the raw income column) and compare. How big is the difference?

> What I saw: 

In [ ]:
raw_income = pd.read_csv("loans.csv").monthly_income_ugx   # the blanks are still blank in this fresh copy
train_median_income = np.nan   # your code here: the median of raw_income for the rows in X_train (raw_income.loc[X_train.index])
print("training rows only:", train_median_income, " all loans:", median_income)

check("median income of the training loans", safe(lambda: train_median_income), 643200.0, 1)

---
## Step 7: The baseline model

The simplest possible model predicts the majority class for every loan. Every real model has to beat it.

**PREDICT first.** What accuracy will the baseline model get on the test set? How many of the real defaults will it catch?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
baseline_model = DummyClassifier(strategy="constant", constant=1)   # your code here: DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)
y_base = baseline_model.predict(X_test)
base_acc = np.nan   # your code here: accuracy_score(y_test, y_base)
base_recall = np.nan   # your code here: recall_score(y_test, y_base)
print("baseline accuracy", round(base_acc, 4), " recall", base_recall)

check("baseline accuracy", safe(lambda: base_acc), 0.8745, 0.001)
check("baseline recall", safe(lambda: base_recall), 0, 0.001)

**READ the output.** The accuracy looks high. Why is it the accuracy paradox?

> My answer: 

---
## Step 8: Fit a first model and read what accuracy hides

A decision tree with max_depth=3 learns from the training set. We predict the test set once and compare with the baseline model. Expected on the course split: accuracy 0.8875 and 41 defaults caught. The checks accept the range seen over ten other splits (0.882 to 0.893), because a library version may break a tie differently.

**PREDICT first.** Will the tree beat the baseline accuracy by a lot, or by about one point? Will it catch most of the defaults?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
model = DecisionTreeClassifier(max_depth=1, random_state=42)   # your code here: DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
y_pred = np.zeros(len(X_test), dtype=int)   # your code here: model.predict(X_test)
model_acc = accuracy_score(y_test, y_pred)
model_recall = np.nan   # your code here: recall_score(y_test, y_pred)
print("tree accuracy", round(model_acc, 4), " recall", round(model_recall, 3))
print(confusion_matrix(y_test, y_pred))   # rows: truth 0 and 1. columns: predicted 0 and 1

check("tree accuracy in the expected range", safe(lambda: model_acc), 0.8878, 0.0103)
check("tree recall in the expected range", safe(lambda: model_recall), 0.1693, 0.0798)
check("tree beats the baseline model", safe(lambda: float(model_acc > base_acc)), 1, 0)

**READ the output.** How many defaults did the tree catch (bottom right of the matrix) and how many did it miss (bottom left)? Is it good enough to lend by?

> My answer: 

**CHANGE one thing.** Fit the tree with max_depth=1, then max_depth=6, and print the accuracy and recall each time. Do not choose a depth by the test score: Session 17 shows how cross-validation on the training loans chooses it.

> What I saw: 

In [ ]:
for depth in (1, 3, 6):
    m = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    p = m.predict(X_test)
    print("max_depth", depth, " accuracy", round(accuracy_score(y_test, p), 4), " recall", round(recall_score(y_test, p), 3))

**CHANGE one thing.** Add the leaking column collection_calls to the features and fit again. What happens to the accuracy? Why is that useless at approval time?

> What I saw: 

In [ ]:
leak_train = X_train.assign(collection_calls=loans.loc[X_train.index, "collection_calls"])
leak_test = X_test.assign(collection_calls=loans.loc[X_test.index, "collection_calls"])
leak_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(leak_train, y_train)
leak_acc = np.nan   # your code here: accuracy_score of leak_model on leak_test
print("accuracy with the leaking column", round(leak_acc, 4))

check("the leak makes accuracy suspiciously high", safe(lambda: float(leak_acc > 0.95)), 1, 0)

---
## Step 9: Keep a prepared file and write your note

Save the 12 features and the target as loans_prepared.csv so you can reload it instead of repeating the preparation.

In [ ]:
prepared = X.copy()
prepared["default"] = y
prepared.to_csv("loans_prepared.csv", index=False)
print(pd.read_csv("loans_prepared.csv").shape)

check("prepared file has 13 columns", pd.read_csv("loans_prepared.csv").shape[1], 13, 0)
check("prepared file has 8,000 rows", pd.read_csv("loans_prepared.csv").shape[0], 8000, 0)

## Your three-sentence note

Double-click and write:

1. Did the first model beat the baseline model, and by how many points of accuracy?
2. How many of the real defaults did it catch?
3. What does accuracy hide, and what would you try next?

> My note: 

In [ ]:
show_checks()